# 01 - Plain text NER (Task 1) and LLM chat (Task 4)

Pipeline: `validation.jsonl` -> GLiNER -> predictions (`results/`) -> strict / relaxed evaluation.

- Task 1: `gliner_small-v2.5`, `gliner_small-v2.5` + label descriptions, `gliner_large-v2.5`.
- Task 4: zero-shot NER with a chat LLM (no API) on `validation_small.jsonl`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))  # makes `import ner` work from /notebooks

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import (
    LABEL_DESCRIPTIONS,
    get_dataset_labels,
    load_gliner_model,
    predict_dataset,
)
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = ROOT / "data" / "plain" / "validation.jsonl"
VALIDATION_SMALL_PATH = ROOT / "data" / "plain" / "validation_small.jsonl"
RESULTS_PLAIN = ROOT / "results" / "plain"
RESULTS_LLM = ROOT / "results" / "llm"
THRESHOLD = 0.5

In [ ]:
gold = load_jsonl(VALIDATION_PATH)
assert gold, f"{VALIDATION_PATH} is empty: copy the course data/ folder into the project first."
LABELS = get_dataset_labels(gold)
print(f"{len(gold)} gold documents | labels: {LABELS}")

## What are the `spans` returned by `predict_entities`?

Each predicted entity is a dict with:

- `start`: index of the first character of the mention (0-based, inclusive).
- `end`: index after the last character (exclusive), so `text[start:end] == entity["text"]`.
- `text`: the mention extracted from the document.
- `label`: the predicted entity type.
- `score`: model confidence (0-1); entities below `threshold` are discarded.

Strict evaluation needs an exact match of `(start, end, label)`.

In [ ]:
model_small = load_gliner_model("gliner-community/gliner_small-v2.5")

sample = gold[0]
for ent in model_small.predict_entities(sample["text"], LABELS, threshold=THRESHOLD)[:8]:
    assert sample["text"][ent["start"]:ent["end"]] == ent["text"]
    print(ent["start"], ent["end"], ent["label"], round(ent["score"], 2), repr(ent["text"]))

## Task 1.1 - `gliner_small-v2.5` (no label descriptions)

In [ ]:
SMALL_PATH = RESULTS_PLAIN / "gliner_small_validation.jsonl"

preds = predict_dataset(model_small, gold, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, SMALL_PATH)

strict_small = evaluate_ner_strict(VALIDATION_PATH, SMALL_PATH)
relaxed_small = evaluate_ner_relaxed(VALIDATION_PATH, SMALL_PATH)

## Task 1.2 - `gliner_small-v2.5` + label descriptions

The model receives a natural-language description of each label instead of the bare label name;
the original label name is restored in the output (see `ner.inference.predict_entities`).

In [ ]:
SMALL_DESC_PATH = RESULTS_PLAIN / "gliner_small_descriptions_validation.jsonl"
descriptions = {lbl: LABEL_DESCRIPTIONS.get(lbl, lbl) for lbl in LABELS}

preds = predict_dataset(model_small, gold, labels=descriptions, threshold=THRESHOLD)
save_responses_to_jsonl(preds, SMALL_DESC_PATH)

strict_small_desc = evaluate_ner_strict(VALIDATION_PATH, SMALL_DESC_PATH)
relaxed_small_desc = evaluate_ner_relaxed(VALIDATION_PATH, SMALL_DESC_PATH)

## Task 1.3 - `gliner_large-v2.5`

In [ ]:
import gc

del model_small
gc.collect()

LARGE_PATH = RESULTS_PLAIN / "gliner_large_validation.jsonl"
model_large = load_gliner_model("gliner-community/gliner_large-v2.5")

preds = predict_dataset(model_large, gold, labels=LABELS, threshold=THRESHOLD)
save_responses_to_jsonl(preds, LARGE_PATH)

strict_large = evaluate_ner_strict(VALIDATION_PATH, LARGE_PATH)
relaxed_large = evaluate_ner_relaxed(VALIDATION_PATH, LARGE_PATH)

del model_large
gc.collect()

## Task 4 - NER with a chat LLM (no API)

1. Run the next cell: it writes `results/llm/prompt.txt`.
2. Paste the prompt in the chat of an LLM (ChatGPT, Gemini, Claude...).
3. Save the answer (JSONL) in `results/llm/raw_llm_response.txt`.
4. Run the cell after that: it parses the answer, recomputes `start`/`end` on the original text
   (LLMs cannot count characters) and evaluates against `validation_small.jsonl`.

In [ ]:
from ner.llm import build_llm_prompt

gold_small = load_jsonl(VALIDATION_SMALL_PATH)
assert gold_small, f"{VALIDATION_SMALL_PATH} is empty"

RESULTS_LLM.mkdir(parents=True, exist_ok=True)
prompt = build_llm_prompt(gold_small, descriptions)
(RESULTS_LLM / "prompt.txt").write_text(prompt, encoding="utf-8")
print(f"{len(gold_small)} documents -> {RESULTS_LLM / 'prompt.txt'} ({len(prompt)} characters)")

In [ ]:
from ner.llm import llm_records_to_predictions, parse_llm_response

RAW_PATH = RESULTS_LLM / "raw_llm_response.txt"
LLM_PATH = RESULTS_LLM / "llm_validation.jsonl"

parsed = parse_llm_response(RAW_PATH.read_text(encoding="utf-8"))
llm_preds = llm_records_to_predictions(parsed, gold_small, valid_labels=LABELS)
save_responses_to_jsonl(llm_preds, LLM_PATH)

n_raw = sum(len(r.get("entities", [])) for r in parsed)
n_kept = sum(len(p["entities"]) for p in llm_preds)
print(f"{len(parsed)}/{len(gold_small)} documents answered | entities kept after alignment: {n_kept}/{n_raw}")

strict_llm = evaluate_ner_strict(VALIDATION_SMALL_PATH, LLM_PATH)
relaxed_llm = evaluate_ner_relaxed(VALIDATION_SMALL_PATH, LLM_PATH)

## Results summary (all experiments found in `results/`)

In [ ]:
from ner.report import build_results_table

table, missing = build_results_table(ROOT)
print(table)
if missing:
    print("\nNot evaluated yet:", *missing, sep="\n  - ")